# FITS to CSV

Converts a FITS table to CSV. Handles byte-string columns and skips multidimensional columns that
cannot be flattened into CSV.

Run it once for each Gentile Fusillo catalogue file in `data/raw/`, setting `fits_path` each time:

- `gaiaedr3_wd_sdssspec.fits.gz` -> `data/gaiaedr3_wd_sdssspec.csv`
- `gaiaedr3_wd_main.fits.gz` -> `data/gaiaedr3_wd_main.csv`

The output names are the ones `02_wd_classification.ipynb` loads.

In [ ]:
# pip install astropy pandas  (uncomment if needed)
from pathlib import Path
import pandas as pd
from astropy.table import Table
from astropy.io import fits

In [ ]:
fits_path = "../data/raw/gaiaedr3_wd_sdssspec.fits.gz"   # input FITS file
csv_path  = None               # output; None -> data/<name>.csv
hdu_index = None               # None -> auto-pick first table HDU

fits_path = Path(fits_path)
if csv_path is None:
    # with_suffix would only swap ".gz" (giving "x.fits.csv"), so strip the FITS extensions by hand
    stem = fits_path.name.removesuffix(".gz").removesuffix(".fits")
    csv_path = Path("../data") / f"{stem}.csv"

In [ ]:
# Inspect the file. FITS can hold several extensions; you want a BinTableHDU.
with fits.open(fits_path) as hdul:
    hdul.info()
    table_hdus = [i for i, h in enumerate(hdul)
                  if isinstance(h, (fits.BinTableHDU, fits.TableHDU))]
print("\nTable HDUs at indices:", table_hdus)

In [ ]:
# Load the table. Table.read picks the first table HDU by default;
# pass hdu_index to override.
if hdu_index is None:
    tbl = Table.read(fits_path)
else:
    tbl = Table.read(fits_path, hdu=hdu_index)

print(f"{len(tbl)} rows, {len(tbl.colnames)} columns")
print(tbl.colnames)

In [ ]:
# Drop multidimensional / vector columns (e.g. flux arrays per row).
# These cannot live in a flat CSV cell. Comment this out if you want to
# keep them serialised as strings instead.
multidim = [c for c in tbl.colnames if len(tbl[c].shape) > 1]
if multidim:
    print("Dropping multidimensional columns:", multidim)
    tbl.remove_columns(multidim)

In [ ]:
# Convert to pandas. to_pandas() masks fill values to NaN automatically.
df = tbl.to_pandas()

# FITS string columns often come back as bytes (b'G2V'); decode to str.
for col in df.select_dtypes([object]).columns:
    df[col] = df[col].apply(lambda v: v.decode("utf-8", "ignore")
                            if isinstance(v, (bytes, bytearray)) else v)
df.head()

In [ ]:
df.to_csv(csv_path, index=False)
print(f"Wrote {len(df)} rows to {csv_path}")